# ETL project for GDP data using Web scraping

Objectives:
- Write a data extraction function to retrieve the relevant information from the required URL.

- Transform the available GDP information into 'Billion USD' from 'Million USD'.

- Load the transformed information to the required CSV file and as a database file.

- Run the required query on the database.

- Log the progress of the code with appropriate timestamps.

### imports

In [84]:
# imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime
import sqlite3

### Logging messages

In [85]:

# log function
def log_progress(message): 
    timestamp_format = '%Y-%h-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second 
    now = datetime.now() # get current timestamp 
    timestamp = now.strftime(timestamp_format) 
    with open(log_file,"a") as f: 
        f.write(timestamp + ',' + message + '\n') 


### Extract

In [87]:
# web scraping
def scrape_gdp_data(url):
    response = requests.get(url)
    log_progress(f' request sent, response status : {response.status_code}')
    return response

# parse html to dataframe
def parse_gdp_data(response):
    parsed_html = BeautifulSoup(response.text, 'html.parser')
    log_progress(f' HTML parsed successfully, using BeautifulSoup')
    return parsed_html

# extract GDP data from html
def extract_gdp_table(parsed_html):
    tables = parsed_html.find_all('table')
    gdp_table = tables[2]
    log_progress(f' GDP table extracted from html')
    return gdp_table


In [88]:
# extract columns from table and create a dataframe
def extract_gdp_columns(gdp_table):
    gdp_dict = {'Country': [], 'GDP': []}

    for tr in gdp_table.find_all('tr')[3:]:
        if len(tr) != 0 : # remove empty rows
            cells = tr.find_all('td')
            if cells[2].text.strip() != '—': # remove rows with missing GDP data
                gdp_dict['Country'].append(cells[0].text.strip())
                gdp_dict['GDP'].append(cells[2].text.strip())

    log_progress(f' Extracted Country and GDP columns from html table')
    return  pd.DataFrame(gdp_dict) 

### Transfrom

In [89]:
# transform GDP data (remove commas, convert to billions, round to 2 decimal places)
def transform_gdp_data(gdp_df):     
    gdp_df['GDP'] = gdp_df['GDP'].str.replace(',', '').astype(float) # remove commas and convert to float
    gdp_df['GDP_USD_billions'] = np.round(gdp_df['GDP']/1000, 2) # convert to billions
    gdp_df = gdp_df.drop(columns=['GDP']) # drop original GDP column
    log_progress(f' dataframe Transformed (millions to billions + renaming)')
    return gdp_df

### Load

load to csv

In [90]:
def load_df_to_csv(gdp_df, output_csv):
    gdp_df.to_csv(output_csv, index=False)
    log_progress(f' dataframe loaded to csv file : {output_csv}')

load to db (sqlite database)

In [91]:
def load_to_db(df, sql_connection, table_name):
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)
    log_progress(f' dataframe loaded to database table : {table_name}')

### Query

In [99]:
def run_query(query_statement, sql_connection):
	result = pd.read_sql_query(query_statement, sql_connection)
	log_progress(f' query executed : {query_statement}')
	print(result)
	log_progress(f' query result : {result.shape[0]} rows returned')
	return result

## Testing Program

In [102]:
# initializing variables
url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'
output_csv = 'Countries_by_GDP.csv'
db = 'World_Economies.db'
db_table_name = 'Countries_by_GDP'
log_file = 'etl_project_log.txt'

response = scrape_gdp_data(url)
parsed_html = parse_gdp_data(response)
gdp_table = extract_gdp_table(parsed_html)
gdp_df = extract_gdp_columns(gdp_table)
transformed_gdp_df = transform_gdp_data(gdp_df)

load_df_to_csv(transformed_gdp_df, output_csv)

# Connect to the SQLite3 service
conn = sqlite3.connect(db)
load_to_db(transformed_gdp_df, conn, db_table_name)

#running query for countries with GDP > 100 billion USD
query = f'SELECT * FROM {db_table_name} WHERE GDP_USD_billions > 100'
run_query(query, conn)
load_df_to_csv(run_query(query, conn), 'Countries_with_GDP_over_100_billion.csv')

#running query for top 10 countries with GDP
query = f'SELECT * FROM {db_table_name} ORDER BY GDP_USD_billions DESC LIMIT 10'
run_query(query, conn)
load_df_to_csv(run_query(query, conn), 'Top_10_Countries_by_GDP.csv')



conn.close()

          Country  GDP_USD_billions
0   United States          26854.60
1           China          19373.59
2           Japan           4409.74
3         Germany           4308.85
4           India           3736.88
..            ...               ...
64          Kenya            118.13
65         Angola            117.88
66           Oman            104.90
67      Guatemala            102.31
68       Bulgaria            100.64

[69 rows x 2 columns]
          Country  GDP_USD_billions
0   United States          26854.60
1           China          19373.59
2           Japan           4409.74
3         Germany           4308.85
4           India           3736.88
..            ...               ...
64          Kenya            118.13
65         Angola            117.88
66           Oman            104.90
67      Guatemala            102.31
68       Bulgaria            100.64

[69 rows x 2 columns]
          Country  GDP_USD_billions
0   United States          26854.60
1           China 